# revT2V: compare reverse-time methods

All methods run on **one shared U-Net** (`revt2v.infer.MethodBank`) from the **same initial noise `z`**:

| Column | What it is |
|---|---|
| `teacher` | frozen ModelScope teacher, forward time, from `z` |
| `matched_target` | `reverse(teacher(latents = flip(z)))`, the per-sample reference a reverse generator started from `z` should reproduce |
| `attn_rotation` | trained self-rotation + LoRA (up-block temporal attn1) |
| `conv_oracle` | teacher with every temporal Conv3d kernel flipped in time; no training |
| `conv_student` | LoRA on temporal Conv3d (only if its checkpoint exists) |

**Caveat:** `conv_oracle` is an exact time-mirror of the teacher (`unet_flipped(z) == flip(unet(flip(z)))`), so it is equivalent to reversing the teacher's output. It's a **reference / upper bound**, not a learned method.

Scoring is done by `scripts/evaluate.py`, every method vs `matched_target`:
- **FVD**: I3D features vs the matched-target set (lower = better). Noisy with few videos, so directional only.
- **CLIP score**: 100 × mean frame-prompt CLIP cosine (higher = better).
- **VBench**: quality dimensions in custom-input mode, plus `vbench_mean`.
- **Optical flow**: direction cosine (+1 = same motion direction, -1 = opposite) and motion ratio (flow magnitude vs target; **frozen** if < 0.3).

## 1. Setup

In [ ]:
# Change this to your GitHub username
GITHUB_USER = "silentlooop"

!git clone -q https://github.com/{GITHUB_USER}/revT2V.git /content/revT2V 2>/dev/null || git -C /content/revT2V pull -q
%cd /content/revT2V
!pip install -q -e ".[eval]"
# VBench pins transformers==4.33.2 and numpy<2, which would break diffusers, so
# install it without deps plus only what its quality dimensions use. decord has
# no wheels for Python 3.13; scripts/evaluate.py swaps in an imageio stand-in.
!pip install -q --no-deps vbench
!pip install -q openai-clip easydict omegaconf
# no %autoreload: it breaks on Colab's Python 3.13 (removed `imp` module).
# After a git pull, restart the kernel and re-run from the top.

In [ ]:
import os
import torch

HF_TOKEN = ""  # paste your Hugging Face token here
os.environ["HF_TOKEN"] = HF_TOKEN
print("CUDA:", torch.cuda.is_available(), "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no GPU")

In [3]:
HF_REPO = "silentlooop/revt2v-ckpt"

PROMPT_SETS = {
    # fluids / gravity / walking: the original comparison set
    "physics": [
        "A person is using a computer",
        "A person is washing dishes",
        "A person is washing hands",
        "A person is wrestling",
        "A person is writing",
    ]
}
PROMPT_SET = "physics"  # "physics" | "irreversible"
PROMPTS = PROMPT_SETS[PROMPT_SET]
OUT_DIR = f"/content/compare_results/{PROMPT_SET}"  # one folder per set, so runs don't overwrite
VIDEOS_DIR = f"{OUT_DIR}/videos"  # <method>/<name>.mp4 + prompts.json, the layout evaluate.py scores
SEEDS = [0, 1]
GEN = dict(negative_prompt="watermark, text", num_inference_steps=25, guidance_scale=9.0,
           num_frames=16, height=256, width=256)
SHEET_FRAMES = [0, 5, 10, 15]
FPS = 8
os.makedirs(OUT_DIR, exist_ok=True)

## 2. Load the teacher once, then the trained adapters

In [ ]:
from revt2v.infer import MethodBank

bank = MethodBank(hf_repo_id=HF_REPO)
bank.pipeline.set_progress_bar_config(disable=True)

# attn_rotation: tries attn_rotation/latest.pt, then the original root latest.pt.
for method in ["attn_rotation", "conv_student"]:
    try:
        print(method, "loaded" if bank.load_adapter(method) else "-> no checkpoint, skipped")
    except Exception as exc:  # e.g. root checkpoint belongs to another method
        print(method, "-> skipped:", exc)

print("available methods:", bank.methods)

In [ ]:
def initial_noise(seed):
    g = torch.Generator(device="cuda").manual_seed(seed)
    return torch.randn((1, 4, GEN["num_frames"], GEN["height"] // 8, GEN["width"] // 8),
                       generator=g, device="cuda", dtype=bank.pipeline.unet.dtype)

def to_np(video):
    """(F, H, W, C) float tensor in [0, 1] -> float32 numpy."""
    return video.float().clamp(0, 1).cpu().numpy()

## 3. Exactness checks
(a) U-Net level: `unet_flipped(z, t, emb)` vs `flip(unet(flip(z), t, emb))`

In [ ]:
from contextlib import nullcontext
from revt2v.methods.conv_mirror import find_temporal_convs, flip_temporal_convs, set_flip_state
from revt2v.teacher import encode_prompt

unet = bank.pipeline.unet
print(len(find_temporal_convs(unet)), "temporal Conv3d layers")

z = initial_noise(0)
emb, _ = encode_prompt(bank.pipeline, PROMPTS[0])
emb = emb.to(unet.dtype)
t = torch.tensor([500], device="cuda")

adapters_off = bank.student.disable_adapter() if bank.student is not None else nullcontext()
try:
    with torch.no_grad(), adapters_off:
        mirrored = unet(z.flip(2), t, encoder_hidden_states=emb, return_dict=False)[0].flip(2)
        flip_temporal_convs(unet)
        flipped = unet(z, t, encoder_hidden_states=emb, return_dict=False)[0]
finally:
    set_flip_state(unet, ())

print(f"max |unet_flipped(z) - flip(unet(flip(z)))| = {(flipped.float() - mirrored.float()).abs().max().item():.3e}")

(b) Video level: `conv_oracle(latents=z)` vs `matched_target(prompt, z)`

In [ ]:
z = initial_noise(0)
oracle = to_np(bank.generate(PROMPTS[0], "conv_oracle", latents=z, **GEN)["video"])
target = to_np(bank.matched_target(PROMPTS[0], z, **GEN)["video"])
print(f"mean pixel diff (0-1 scale): {abs(oracle - target).mean():.4f}")

## 4. Load `scripts/evaluate.py`

In [ ]:
import importlib.util
import numpy as np

# scripts/ isn't a package (and `evaluate` would clash with Hugging Face's
# `evaluate` library), so load the file directly.
spec = importlib.util.spec_from_file_location("revt2v_evaluate", "scripts/evaluate.py")
ev = importlib.util.module_from_spec(spec)
spec.loader.exec_module(ev)

## 5. Generate every method from the same noise

In [ ]:
from diffusers.utils import export_to_video
from IPython.display import Video, display

COLUMNS = [m for m in ["teacher", "matched_target", "attn_rotation", "conv_oracle", "conv_student"]
           if m == "matched_target" or m in bank.methods]

# Every method + matched_target from the same noise per (prompt, seed); names are p<idx>_s<seed>.
videos, names = ev.generate_videos(bank, PROMPTS, SEEDS, [m for m in COLUMNS if m != "matched_target"], GEN)
ev.save_video_set(videos, names, VIDEOS_DIR, fps=FPS)

os.makedirs(f"{OUT_DIR}/side_by_side", exist_ok=True)
for name, prompt in names.items():
    side_by_side = np.clip(np.concatenate([videos[m][name] for m in COLUMNS], axis=2), 0, 1)
    # export_to_video expects float frames in [0, 1], not uint8
    path = export_to_video(list(side_by_side), f"{OUT_DIR}/side_by_side/{name}.mp4", fps=FPS)
    print(f"{name}: {prompt}\n    " + " | ".join(COLUMNS))
    display(Video(path, embed=True, width=256 * len(COLUMNS)))

## 6. Scores from `scripts/evaluate.py` (vs matched target)

In [ ]:
import pandas as pd

# The first run downloads CLIP ViT-L/14, I3D and the VBench models (a few GB).
# Writes metrics.csv, summary.csv and summary.json to OUT_DIR.
result = ev.score_videos_dir(VIDEOS_DIR, output_dir=OUT_DIR)

summary = pd.DataFrame(result["summary"]).set_index("method")
display(summary.round(3))
display(pd.DataFrame(result["rows"]).round(3))

## 7. Frame sheets (rows = methods, columns = frames)

In [ ]:
import matplotlib.pyplot as plt

for p_idx, prompt in enumerate(PROMPTS):
    name = f"p{p_idx}_s{SEEDS[0]}"
    fig, axes = plt.subplots(len(COLUMNS), len(SHEET_FRAMES),
                             figsize=(2.2 * len(SHEET_FRAMES), 2.2 * len(COLUMNS)), squeeze=False)
    for r, method in enumerate(COLUMNS):
        for c, f in enumerate(SHEET_FRAMES):
            ax = axes[r][c]
            ax.imshow(np.clip(videos[method][name][f], 0, 1))
            ax.set_xticks([]); ax.set_yticks([])
            if r == 0: ax.set_title(f"frame {f}")
            if c == 0: ax.set_ylabel(method)
    fig.suptitle(f"[{p_idx}] {prompt} (seed {SEEDS[0]})", fontsize=9)
    fig.tight_layout()
    fig.savefig(f"{OUT_DIR}/p{p_idx}_frame_sheet.png", dpi=100)
    plt.show()

## 8. Zip and download

In [ ]:
import shutil
from google.colab import files

archive = shutil.make_archive(f"/content/compare_results_{PROMPT_SET}", "zip", OUT_DIR)
print(archive)
files.download(archive)